# 公司图纸伪标注生成流水线
## CubiCasa → DINOv2+LoRA → SAM2精化 → VLM语义补全 → SVG

```
目标：用无标注的公司图纸生成高质量 SVG 伪标注，用于后续 LoRA 微调

四步流水线：

  Step 1  CubiCasa 训练 DINOv2+LoRA
          → 输出: best_model_dinov2_lora.pth
          → 能力: 在公司图纸上生成粗糙的 wall/door/window 初始 mask

  Step 2  SAM2 精化 (Point Prompt)
          初始 mask 采样正负点 → SAM2 → 精细化边界
          → 输出: refined_mask.png (每张图)

  Step 3  VLM 语义补全 (Claude claude-sonnet-4)
          图片 + refined_mask → VLM → 门窗类型/朝向/尺寸 JSON
          → 输出: semantic_meta.json (每张图)

  Step 4  SVG 脚本生成
          wall_boxes + openings_meta → Shrinking → 标准 SVG
          → 输出: pseudo_label.svg (每张图，CubiCasa 兼容格式)
```

### 为什么选 DINOv2 而不是 SAM2 做第一步？
SAM2 是交互式分割，没有墙体的语义概念，直接用效果差。  
DINOv2 通过 LoRA 学到了 CubiCasa 里墙体的外观特征，能给 SAM2 提供高质量的初始点 prompt。  
两者分工：**DINOv2 负责在哪里，SAM2 负责边界在哪**。

## Cell 0 · 环境配置

In [1]:
import warnings; warnings.filterwarnings('ignore')
import os, sys, gc, json, time, random, logging
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict
from collections import OrderedDict

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.ops import FeaturePyramidNetwork
import albumentations as A
from albumentations.pytorch import ToTensorV2
import mlflow
from tqdm.notebook import tqdm

try:
    import timm
    HAS_TIMM = True
except ImportError:
    HAS_TIMM = False
    print('[!] pip install timm')

try:
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    HAS_SAM2 = True
except ImportError:
    HAS_SAM2 = False
    print('[!] pip install segment-anything-2')

try:
    import anthropic
    HAS_ANTHROPIC = True
except ImportError:
    HAS_ANTHROPIC = False
    print('[!] pip install anthropic')

# ── 路径 ──
if os.path.exists('/workspace/production_3d'):
    PROJECT_ROOT  = '/workspace/production_3d'
    CUBICASA_ROOT = '/workspace/CubiCasa5k'
    DATA_FOLDER   = '/workspace/data/cubicasa5k/'
    COMPANY_DIR   = '/workspace/data/company_floorplans/'
    SAM2_CKPT     = '/workspace/checkpoints/sam2_hiera_large.pt'
elif os.path.exists('/content'):
    PROJECT_ROOT  = '/content/production_3d'
    CUBICASA_ROOT = '/content/CubiCasa5k'
    DATA_FOLDER   = '/content/data/cubicasa5k/'
    COMPANY_DIR   = '/content/data/company_floorplans/'
    SAM2_CKPT     = '/content/checkpoints/sam2_hiera_large.pt'
else:
    PROJECT_ROOT  = '.'
    CUBICASA_ROOT = r'E:\JOB\CubiCasa5k'
    DATA_FOLDER   = r'C:/Users/kawayi_yaling/.cache/kagglehub/datasets/qmarva/cubicasa5k/versions/4/cubicasa5k/cubicasa5k/'
    COMPANY_DIR   = './data/company_floorplans/'
    SAM2_CKPT     = './checkpoints/sam2_hiera_large.pt'

CKPT_DIR       = os.path.join(PROJECT_ROOT, 'checkpoints_dinov2_lora')
PSEUDO_OUT_DIR = os.path.join(PROJECT_ROOT, 'pseudo_labels')
MLFLOW_DIR     = os.path.join(PROJECT_ROOT, 'mlruns')
for d in [CKPT_DIR, PSEUDO_OUT_DIR, MLFLOW_DIR]:
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, CUBICASA_ROOT)
os.chdir(CUBICASA_ROOT)

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s [%(levelname)s] %(message)s',
                    datefmt='%H:%M:%S')
logger = logging.getLogger(__name__)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'✓ 环境就绪  device={DEVICE}')
print(f'  HAS_TIMM={HAS_TIMM}  HAS_SAM2={HAS_SAM2}  HAS_ANTHROPIC={HAS_ANTHROPIC}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

✓ 环境就绪  device=cuda
  HAS_TIMM=True  HAS_SAM2=True  HAS_ANTHROPIC=True
  GPU: NVIDIA RTX A6000


## Cell 1 · 配置

In [2]:
@dataclass
class PseudoLabelConfig:
    # ══════════════════════════════════════════════════════
    # 数据集版本（三选一）
    # 'hq'      → 只用 high_quality
    # 'hq_arch' → 只用 high_quality_architectural
    # 'combined'→ 两者合集（推荐）
    # ══════════════════════════════════════════════════════
    dataset_version: str = 'combined'

    # ── DINOv2 + LoRA ──
    dinov2_model:     str       = 'vit_large_patch14_dinov2'
    lora_r:           int       = 16
    lora_alpha:       int       = 32
    lora_dropout:     float     = 0.1
    lora_target:      List[str] = field(default_factory=lambda: ['qkv', 'proj'])
    fpn_out_channels: int       = 256
    seg_num_classes:  int       = 2    # background=0, wall=1
    det_num_classes:  int       = 3    # background=0, door=1, window=2

    # ── 数据路径 ──
    data_folder: str = DATA_FOLDER

    # ── 类别 ID（CubiCasa5k 约定）──
    wall_class_id:   int = 2
    door_class_id:   int = 2
    window_class_id: int = 1
    min_bbox_area:   int = 100

    # ── 图像处理 ──
    tile_size:    int   = 518   # DINOv2 patch_size=14，必须是14的倍数 (37×14=518)
    tile_overlap: int   = 56    # 14的倍数，stride=462=33×14
    norm_mean:    Tuple = (0.485, 0.456, 0.406)
    norm_std:     Tuple = (0.229, 0.224, 0.225)

    # ── 训练超参 ──
    batch_size:        int   = 4
    num_workers:       int   = 2
    max_epochs:        int   = 50
    learning_rate:     float = 5e-5
    weight_decay:      float = 1e-4
    warmup_epochs:     int   = 3
    use_amp:           bool  = True
    grad_clip:         float = 3.0
    wall_class_weight: float = 5.0
    dice_weight:       float = 0.5
    bce_weight:        float = 0.5

    # ── 断点续训 ──
    # 填入要恢复的 checkpoint 路径，None = 从头开始
    # 例: resume_from = '/workspace/.../hq/hq_bs4_lr5e-5_ep020_iou0.7823.pth'
    resume_from: str = None

    # ── 检查点目录：按 dataset_version 自动分子目录 ──
    @property
    def checkpoint_dir(self) -> str:
        d = os.path.join(CKPT_DIR, self.dataset_version)
        os.makedirs(d, exist_ok=True)
        return d

    # ── MLflow：所有版本共用一个 experiment，run_name 含超参 ──
    @property
    def mlflow_experiment(self) -> str:
        return 'dinov2_lora_floorplan'

    @property
    def mlflow_run_name(self) -> str:
        lr_str = f'{self.learning_rate:.0e}'.replace('-0', '-')
        return (f'{self.dataset_version}'
                f'_ep{self.max_epochs}'
                f'_bs{self.batch_size}'
                f'_lr{lr_str}'
                f'_{time.strftime("%m%d_%H%M")}')

    # ── split 文件（由 dataset_version 自动推导）──
    @property
    def train_files(self) -> List[str]:
        return _version_to_files(self.dataset_version, 'train')

    @property
    def val_files(self) -> List[str]:
        return _version_to_files(self.dataset_version, 'val')

    # ── GCS 配置 ──
    gcs_bucket:  str = 'yalingdata'
    gcs_project: str = 'project-d3027d52-508f-4689-899'

    # GCS 模型目录：按 dataset_version / run_tag 自动分子目录，不手动填
    # 结构: models/dinov2_lora/{dataset_version}/{run_tag}/
    @property
    def gcs_model_dir(self) -> str:
        lr_str  = f'{self.learning_rate:.0e}'.replace('-0', '-')
        run_tag = (f'{self.dataset_version}'
                   f'_ep{self.max_epochs}'
                   f'_bs{self.batch_size}'
                   f'_lr{lr_str}')
        return f'models/dinov2_lora/{self.dataset_version}/{run_tag}'

    # ── Step 2: SAM2 精化 ──
    sam2_ckpt:         str   = SAM2_CKPT
    sam2_cfg:          str   = 'sam2_hiera_l.yaml'
    sam2_n_pos_points: int   = 5
    sam2_n_neg_points: int   = 3
    sam2_score_thresh: float = 0.8

    # ── Step 3: VLM 语义补全 ──
    vlm_model:      str = 'claude-sonnet-4-20250514'
    vlm_max_tokens: int = 1024

    # ── Step 4: SVG 生成 ──
    pixels_per_meter:  float = 50.0
    wall_height_m:     float = 2.8
    door_height_m:     float = 2.1
    window_height_m:   float = 1.2
    shrink_iou_thresh: float = 0.85
    min_segment_area:  int   = 200

    # ── 输出 ──
    pseudo_out_dir: str = PSEUDO_OUT_DIR
    company_dir:    str = COMPANY_DIR


# ── split 文件映射 ──
def _version_to_files(version: str, split: str) -> List[str]:
    mapping = {
        'hq':       [f'{split}_hq.txt'],
        'hq_arch':  [f'{split}_hq_arch.txt'],
        'combined': [f'{split}_hq.txt', f'{split}_hq_arch.txt'],
    }
    if version not in mapping:
        raise ValueError(f'dataset_version 必须是 hq/hq_arch/combined，收到: {version}')
    return mapping[version]


CFG = PseudoLabelConfig()

print('✓ 配置完成')
print(f'  dataset_version : {CFG.dataset_version}')
print(f'  train 文件      : {CFG.train_files}')
print(f'  val   文件      : {CFG.val_files}')
print(f'  checkpoint_dir  : {CFG.checkpoint_dir}')
print(f'  mlflow_exp      : {CFG.mlflow_experiment}')
print(f'  mlflow_run_name : {CFG.mlflow_run_name}')
print(f'  resume_from     : {CFG.resume_from}')
print(f'  DINOv2 模型     : {CFG.dinov2_model}')
print(f'  LoRA r={CFG.lora_r}  alpha={CFG.lora_alpha}  target={CFG.lora_target}')
print(f'  gcs_model_dir   : gs://{CFG.gcs_bucket}/{CFG.gcs_model_dir}/')


✓ 配置完成
  dataset_version : combined
  train 文件      : ['train_hq.txt', 'train_hq_arch.txt']
  val   文件      : ['val_hq.txt', 'val_hq_arch.txt']
  checkpoint_dir  : /workspace/production_3d/checkpoints_dinov2_lora/combined
  mlflow_exp      : dinov2_lora_floorplan
  mlflow_run_name : combined_ep50_bs4_lr5e-5_0512_0532
  resume_from     : None
  DINOv2 模型     : vit_large_patch14_dinov2
  LoRA r=16  alpha=32  target=['qkv', 'proj']
  gcs_model_dir   : gs://yalingdata/models/dinov2_lora/combined/combined_ep50_bs4_lr5e-5/


## Cell 2 · Step 1A：DINOv2 编码器 + LoRA 注入

In [3]:
# ══════════════════════════════════════════════════════════════
# DINOv2 ViT-L 编码器
# 从不同深度的 block 抽取中间特征，模拟多尺度输出供 FPN 使用
# ══════════════════════════════════════════════════════════════

class DINOv2Encoder(nn.Module):
    """DINOv2 ViT-L/14 多尺度特征提取器
    从 4 个深度层级抽取中间特征，模拟 FPN 的多尺度输入
    ViT-L: 24 个 block，embed_dim=1024
    抽取层: [5, 11, 17, 23]  对应 1/4, 2/4, 3/4, 4/4 深度
    """
    EXTRACT_LAYERS = [5, 11, 17, 23]

    def __init__(self, model_name: str = 'vit_large_patch14_dinov2'):
        super().__init__()
        if not HAS_TIMM:
            raise RuntimeError('pip install timm')
        # dynamic_img_size=True: 让 timm 接受任意 patch_size 倍数的输入
        # 不再固定为默认的 518x518，配合 tile_size=518 使用
        self.vit = timm.create_model(
            model_name,
            pretrained       = True,
            dynamic_img_size = True,   # 不固定输入尺寸
        )
        self.embed_dim    = self.vit.embed_dim   # ViT-L: 1024
        self.out_channels = [self.embed_dim] * 4
        self._features: Dict[int, torch.Tensor] = {}

        for layer_idx in self.EXTRACT_LAYERS:
            self.vit.blocks[layer_idx].register_forward_hook(
                self._make_hook(layer_idx)
            )

    def _make_hook(self, layer_idx: int):
        def hook(module, input, output):
            self._features[layer_idx] = output
        return hook

    def forward(self, x: torch.Tensor) -> OrderedDict:
        B, C, H, W = x.shape
        self._features.clear()
        _ = self.vit(x)

        patch_h = H // 14
        patch_w = W // 14
        result  = OrderedDict()

        for i, layer_idx in enumerate(self.EXTRACT_LAYERS):
            feat = self._features[layer_idx]          # (B, N+1, D)
            feat = feat[:, 1:, :].permute(0, 2, 1)   # (B, D, N)
            feat = feat.reshape(B, -1, patch_h, patch_w)
            # 浅层特征下采样模拟低分辨率
            if i < 3:
                scale = 2 ** (3 - i)
                feat  = F.avg_pool2d(feat, kernel_size=scale, stride=scale)
            result[str(i)] = feat
        return result


# ══════════════════════════════════════════════════════════════
# LoRA 注入（peft inject_adapter_in_model）
#
# 为什么不用 get_peft_model？
#   get_peft_model 包成 PeftModelForFeatureExtraction，
#   其 forward 强制传 input_ids / attention_mask 等 NLP 参数，
#   和自定义 DINOv2Encoder.forward(x) 不兼容，直接 TypeError。
#
# inject_adapter_in_model 是 peft 底层 API：
#   只替换目标 Linear 层，不碰 forward，不改接口
#   权重命名遵循 peft 标准（lora_A / lora_B）
#   可用 peft 工具只存 LoRA 增量（几 MB 而非几百 MB）
# ══════════════════════════════════════════════════════════════

try:
    from peft import LoraConfig, inject_adapter_in_model
    HAS_PEFT = True
except ImportError:
    HAS_PEFT = False
    print('[!] pip install peft')


def inject_lora(encoder: nn.Module, cfg) -> nn.Module:
    """
    用 peft inject_adapter_in_model 注入 LoRA，然后冻结非 LoRA 参数。

    freeze + inject 合并为一步，调用方不需要再单独调用 freeze_encoder。
    """
    if not HAS_PEFT:
        raise RuntimeError('pip install peft')

    lora_config = LoraConfig(
        r              = cfg.lora_r,
        lora_alpha     = cfg.lora_alpha,
        lora_dropout   = cfg.lora_dropout,
        target_modules = cfg.lora_target,   # ['qkv', 'proj']
        bias           = 'none',
    )
    # 只替换目标 Linear，不包装 forward
    encoder = inject_adapter_in_model(lora_config, encoder)

    # 冻结所有非 LoRA 参数
    for name, p in encoder.named_parameters():
        if 'lora_' not in name:
            p.requires_grad = False

    total     = sum(p.numel() for p in encoder.parameters())
    trainable = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
    logger.info(
        f'LoRA 注入 (peft): {trainable/1e6:.2f}M / {total/1e6:.1f}M '
        f'可训练 ({trainable/total*100:.1f}%)'
    )
    return encoder


def save_lora_weights(encoder: nn.Module, save_path: str):
    """只保存 LoRA 增量权重（几 MB），不保存冻结的原始权重"""
    lora_state = {k: v for k, v in encoder.state_dict().items() if 'lora_' in k}
    torch.save(lora_state, save_path)
    size_mb = sum(v.numel() * v.element_size() for v in lora_state.values()) / 1e6
    logger.info(f'LoRA 增量已保存: {save_path}  ({size_mb:.1f} MB  {len(lora_state)} 个张量)')


def load_lora_weights(encoder: nn.Module, load_path: str) -> nn.Module:
    """加载 LoRA 增量权重（strict=False 忽略冻结的原始权重）"""
    lora_state   = torch.load(load_path, map_location='cpu')
    missing, _   = encoder.load_state_dict(lora_state, strict=False)
    lora_missing = [k for k in missing if 'lora_' in k]
    if lora_missing:
        logger.warning(f'以下 LoRA 权重未加载: {lora_missing}')
    logger.info(f'LoRA 权重加载完成: {load_path}')
    return encoder


print('✓ DINOv2 编码器 + peft LoRA 定义完成')
print(f'  HAS_PEFT = {HAS_PEFT}')


✓ DINOv2 编码器 + peft LoRA 定义完成
  HAS_PEFT = True


## Cell 3 · Step 1B：分割头 + 完整模型组装

In [4]:
from torchvision.models.detection.rpn import AnchorGenerator, RPNHead, RegionProposalNetwork
from torchvision.models.detection.roi_heads import RoIHeads
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import MultiScaleRoIAlign
from torchvision.models.detection.image_list import ImageList


class SegHead(nn.Module):
    """FPN + 轻量 decoder → wall mask"""
    def __init__(self, in_ch_list, fpn_ch=256, num_classes=2):
        super().__init__()
        self.fpn     = FeaturePyramidNetwork(in_ch_list, fpn_ch)
        self.decoder = nn.Sequential(
            nn.Conv2d(fpn_ch, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.Conv2d(128,  64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(True),
            nn.Conv2d(64, num_classes, 1),
        )
    def forward(self, features, input_size):
        fpn_out     = self.fpn(features)
        target_size = fpn_out['0'].shape[-2:]
        fused       = fpn_out['0']
        for k in ['1', '2', '3']:
            if k in fpn_out:
                fused = fused + F.interpolate(fpn_out[k], target_size, mode='bilinear', align_corners=False)
        logits = self.decoder(fused)
        return F.interpolate(logits, input_size, mode='bilinear', align_corners=False)


class DINOv2LoRAModel(nn.Module):
    """
    DINOv2 ViT-L + LoRA + FPN 分割头 + Faster R-CNN 检测头
    在 CubiCasa 上训练，用于生成公司图纸的初始 mask
    """
    def __init__(self, cfg: PseudoLabelConfig):
        super().__init__()
        # ── 编码器: peft inject_adapter_in_model（不包装 forward）──
        self.encoder = DINOv2Encoder(cfg.dinov2_model)
        self.encoder = inject_lora(self.encoder, cfg)   # freeze + inject 合并
        in_ch = self.encoder.out_channels   # [1024, 1024, 1024, 1024]

        # ── 分割头 ──
        self.seg_head = SegHead(in_ch, cfg.fpn_out_channels, cfg.seg_num_classes)

        # ── 检测头 ──
        fpn_ch = cfg.fpn_out_channels
        self.det_fpn = FeaturePyramidNetwork(in_ch, fpn_ch)
        rpn_anchor   = AnchorGenerator(
            sizes=((16,),(32,),(64,),(128,)),
            aspect_ratios=((0.5,1.0,2.0),)*4
        )
        self.rpn = RegionProposalNetwork(
            rpn_anchor, RPNHead(fpn_ch, rpn_anchor.num_anchors_per_location()[0]),
            fg_iou_thresh=0.7, bg_iou_thresh=0.3,
            batch_size_per_image=256, positive_fraction=0.5,
            pre_nms_top_n={'training':2000,'testing':1000},
            post_nms_top_n={'training':2000,'testing':300},
            nms_thresh=0.7,
        )
        self.roi_heads = RoIHeads(
            MultiScaleRoIAlign(['0','1','2','3'], 7, 2),
            nn.Sequential(nn.Flatten(), nn.Linear(fpn_ch*49,1024), nn.ReLU(True),
                          nn.Linear(1024,1024), nn.ReLU(True)),
            FastRCNNPredictor(1024, cfg.det_num_classes),
            fg_iou_thresh=0.5, bg_iou_thresh=0.5,
            batch_size_per_image=512, positive_fraction=0.25,
            bbox_reg_weights=None,
            score_thresh=0.05, nms_thresh=0.5, detections_per_img=100,
        )

    def forward(self, images, targets=None):
        input_size   = images.shape[-2:]
        features     = self.encoder(images)
        seg_logits   = self.seg_head(features, input_size)
        det_features = self.det_fpn(features)
        img_list     = ImageList(images, [images.shape[-2:]]*images.shape[0])

        if self.training and targets is not None:
            proposals, rpn_losses = self.rpn(img_list, det_features, targets)
            _, roi_losses = self.roi_heads(det_features, proposals, [images.shape[-2:]]*images.shape[0], targets)
            return {'seg_logits': seg_logits, 'det_losses': {**rpn_losses, **roi_losses}}
        else:
            proposals, _ = self.rpn(img_list, det_features, None)
            det_out, _   = self.roi_heads(det_features, proposals, [images.shape[-2:]]*images.shape[0], None)
            return {'seg_logits': seg_logits, 'det_outputs': det_out}


print('✓ DINOv2LoRAModel 定义完成')
print('初始化示例 (不实际运行):')
print('  model = DINOv2LoRAModel(CFG).to(DEVICE)')

✓ DINOv2LoRAModel 定义完成
初始化示例 (不实际运行):
  model = DINOv2LoRAModel(CFG).to(DEVICE)


## Cell 4 · Step 1C：CubiCasa 数据集 + 损失函数

In [5]:
from numpy import genfromtxt


# ══════════════════════════════════════════════════════════════
# 生成子集 split 文件（首次使用改为 True，之后保持 False）
# ══════════════════════════════════════════════════════════════

def generate_hq_split_files(data_folder: str, overwrite: bool = False):
    """
    扫描原始 train/val/test.txt，按子文件夹名生成：
      {split}_hq.txt       ← high_quality 样本
      {split}_hq_arch.txt  ← high_quality_architectural 样本
    combined 不单独生成文件，由 Dataset 同时加载以上两个文件实现。
    """
    SUBFOLDER_MAP = {
        'high_quality':               'hq',
        'high_quality_architectural': 'hq_arch',
    }
    generated = []
    for split in ('train', 'val', 'test'):
        src = os.path.join(data_folder, f'{split}.txt')
        if not os.path.exists(src):
            logger.warning(f'[skip] {src} 不存在')
            continue
        all_folders = genfromtxt(src, dtype='str').tolist()
        if isinstance(all_folders, str):
            all_folders = [all_folders]

        buckets = {tag: [] for tag in SUBFOLDER_MAP.values()}
        for folder in all_folders:
            subfolder = folder.strip('/').split('/')[0]
            tag = SUBFOLDER_MAP.get(subfolder)
            if tag:
                buckets[tag].append(folder)

        for tag, folders in buckets.items():
            out = os.path.join(data_folder, f'{split}_{tag}.txt')
            if os.path.exists(out) and not overwrite:
                logger.info(f'[skip]  {os.path.basename(out):35s} 已存在')
                continue
            with open(out, 'w') as f:
                f.write('\n'.join(folders) + ('\n' if folders else ''))
            generated.append(out)
            logger.info(f'[done]  {os.path.basename(out):35s} {len(folders)} 个样本')
    return generated


RUN_GENERATE_SPLITS = False   # ← 第一次使用时改为 True

if RUN_GENERATE_SPLITS:
    gen = generate_hq_split_files(CFG.data_folder, overwrite=False)
    logger.info(f'生成了 {len(gen)} 个 split 文件')

    # 打印三个版本的样本数
    print('\n三个版本样本数对比：')
    print(f'{"版本":<12} {"split":<8} {"文件":<45} {"样本数":>6}')
    print('─' * 75)
    for version in ('hq', 'hq_arch', 'combined'):
        for split in ('train', 'val'):
            files = _version_to_files(version, split)
            total = 0
            for f in files:
                fpath = os.path.join(CFG.data_folder, f)
                if os.path.exists(fpath):
                    rows = genfromtxt(fpath, dtype='str')
                    total += int(rows.size)
            print(f'{version:<12} {split:<8} {str(files):<45} {total:>6}')
else:
    print('跳过 split 文件生成（RUN_GENERATE_SPLITS=False）')


# ══════════════════════════════════════════════════════════════
# FloorplanDataset：支持三个版本，多文件合并去重
# ══════════════════════════════════════════════════════════════

class FloorplanDataset(Dataset):
    """
    支持三个 dataset_version 的 CubiCasa 数据集
      hq       → 加载 {split}_hq.txt
      hq_arch  → 加载 {split}_hq_arch.txt
      combined → 同时加载两个文件，合并去重后训练
    """

    def __init__(self, cfg: PseudoLabelConfig, split: str = 'train'):
        self.cfg      = cfg
        self.is_train = (split == 'train')
        file_list     = cfg.train_files if split == 'train' else cfg.val_files

        # ── 多文件合并加载 ──
        all_folders = []
        for filename in file_list:
            fpath = os.path.join(cfg.data_folder, filename)
            if not os.path.exists(fpath):
                logger.warning(f'split 文件不存在，跳过: {filename}  '
                               f'（先运行 RUN_GENERATE_SPLITS=True）')
                continue
            rows = genfromtxt(fpath, dtype='str').tolist()
            if isinstance(rows, str):
                rows = [rows]
            all_folders.extend(rows)

        # 去重并保持顺序（combined 两个文件可能有极少量重叠）
        seen, unique = set(), []
        for f in all_folders:
            if f not in seen:
                seen.add(f); unique.append(f)
        self.folders = np.array(unique)

        logger.info(
            f'[{split}] version={cfg.dataset_version}  '
            f'files={file_list}  →  {len(self.folders)} 个样本'
        )

        # ── 数据增强 ──
        if self.is_train:
            self.aug = A.Compose([
                A.RandomRotate90(p=0.75),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.ShiftScaleRotate(scale_limit=(-0.3, 0.5), rotate_limit=10,
                                   border_mode=cv2.BORDER_REFLECT_101, p=0.5),
                A.RandomBrightnessContrast(p=0.4),
                A.GaussNoise(var_limit=(0, 0.02 * 255**2), p=0.3),
                A.ImageCompression(quality_lower=70, quality_upper=95, p=0.2),
                A.Normalize(mean=cfg.norm_mean, std=cfg.norm_std),
                ToTensorV2(),
            ], additional_targets={'mask': 'mask'})
        else:
            self.aug = A.Compose([
                A.Normalize(mean=cfg.norm_mean, std=cfg.norm_std),
                ToTensorV2(),
            ], additional_targets={'mask': 'mask'})

    def __len__(self): return len(self.folders)

    def __getitem__(self, idx):
        try:
            return self._load(self.folders[idx])
        except Exception as e:
            logger.warning(f'{self.folders[idx]}: {e}')
            return self._empty()

    def _load(self, folder):
        from floortrans.loaders.house import House
        folder   = folder.strip('/')
        img_path = os.path.join(self.cfg.data_folder, folder, 'F1_scaled.png')
        svg_path = os.path.join(self.cfg.data_folder, folder, 'model.svg')

        img_bgr = cv2.imread(img_path)
        assert img_bgr is not None, f'图片不存在: {img_path}'
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        h, w    = img_rgb.shape[:2]

        seg       = House(svg_path, h, w).get_segmentation_tensor()
        wall_mask = (seg[0] == self.cfg.wall_class_id).astype(np.uint8)

        boxes, labels = [], []
        for cls_id, lbl in [(self.cfg.door_class_id, 1),
                             (self.cfg.window_class_id, 2)]:
            m = (seg[1] == cls_id).astype(np.uint8)
            for cnt in cv2.findContours(
                    m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]:
                if cv2.contourArea(cnt) < self.cfg.min_bbox_area: continue
                x, y, bw, bh = cv2.boundingRect(cnt)
                boxes.append([x, y, x+bw, y+bh]); labels.append(lbl)

        # 随机 tile
        # tile_size 必须是 14 的倍数（DINOv2 patch_size=14）
        # 默认 518 = 37×14，tile_overlap=56=4×14，stride=462=33×14
        ts     = self.cfg.tile_size
        stride = ts - self.cfg.tile_overlap
        ys = list(range(0, max(h-ts+1, 1), stride))
        xs = list(range(0, max(w-ts+1, 1), stride))
        if not ys or ys[-1]+ts < h: ys.append(max(h-ts, 0))
        if not xs or xs[-1]+ts < w: xs.append(max(w-ts, 0))
        ty = random.choice(list(set(ys)))
        tx = random.choice(list(set(xs)))

        img_t  = img_rgb[ty:ty+ts, tx:tx+ts].copy()
        mask_t = wall_mask[ty:ty+ts, tx:tx+ts].copy()
        th, tw = img_t.shape[:2]
        if th < ts or tw < ts:
            img_t  = cv2.copyMakeBorder(
                img_t, 0, ts-th, 0, ts-tw, cv2.BORDER_REFLECT_101)
            new_m  = np.zeros((ts, ts), dtype=mask_t.dtype)
            new_m[:th, :tw] = mask_t; mask_t = new_m
        img_t  = cv2.resize(img_t,  (ts, ts))
        mask_t = cv2.resize(
            mask_t.astype(np.float32), (ts, ts),
            interpolation=cv2.INTER_NEAREST).astype(np.uint8)

        aug     = self.aug(image=img_t, mask=mask_t)
        img_out = aug['image']
        msk_out = aug['mask'].long()

        tile_boxes, tile_labels = [], []
        for b, l in zip(boxes, labels):
            x1 = np.clip(b[0]-tx, 0, ts); y1 = np.clip(b[1]-ty, 0, ts)
            x2 = np.clip(b[2]-tx, 0, ts); y2 = np.clip(b[3]-ty, 0, ts)
            if (x2-x1) > 5 and (y2-y1) > 5:
                tile_boxes.append([x1, y1, x2, y2]); tile_labels.append(l)

        boxes_t  = torch.tensor(tile_boxes,  dtype=torch.float32) \
                   if tile_boxes  else torch.zeros((0, 4))
        labels_t = torch.tensor(tile_labels, dtype=torch.int64) \
                   if tile_labels else torch.zeros(0, dtype=torch.int64)
        return {'image': img_out, 'mask': msk_out,
                'boxes': boxes_t, 'labels': labels_t}

    def _empty(self):
        ts = self.cfg.tile_size
        return {'image':  torch.zeros(3, ts, ts),
                'mask':   torch.zeros(ts, ts).long(),
                'boxes':  torch.zeros((0, 4)),
                'labels': torch.zeros(0, dtype=torch.int64)}


def collate_fn(batch):
    return {'image':  torch.stack([b['image']  for b in batch]),
            'mask':   torch.stack([b['mask']   for b in batch]),
            'boxes':  [b['boxes']  for b in batch],
            'labels': [b['labels'] for b in batch]}


# ── 损失函数 ──
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth = smooth
    def forward(self, logits, target):
        prob  = torch.softmax(logits, dim=1)[:, 1].reshape(-1)
        tgt   = (target == 1).float().reshape(-1)
        inter = (prob * tgt).sum()
        return 1 - (2*inter + self.smooth) / (prob.sum() + tgt.sum() + self.smooth)


class MultiTaskLoss(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        w = torch.tensor([1.0, cfg.wall_class_weight])
        self.bce  = nn.CrossEntropyLoss(weight=w, ignore_index=255)
        self.dice = DiceLoss()
        self.cfg  = cfg
    def forward(self, seg_logits, seg_targets, det_losses):
        l_bce  = self.bce(seg_logits, seg_targets)
        l_dice = self.dice(seg_logits, seg_targets)
        l_seg  = self.cfg.bce_weight * l_bce + self.cfg.dice_weight * l_dice
        l_det  = sum(det_losses.values()) if det_losses \
                 else torch.tensor(0.0, device=seg_logits.device)
        return {
            'loss_total': l_seg + l_det,
            'loss_seg':   l_seg.item(),
            'loss_bce':   l_bce.item(),
            'loss_dice':  l_dice.item(),
            'loss_det':   l_det.item() if det_losses else 0.0,
        }


print('✓ 数据集 + 损失函数定义完成')


跳过 split 文件生成（RUN_GENERATE_SPLITS=False）
✓ 数据集 + 损失函数定义完成


## Cell 5 · Step 1D：训练主循环

In [7]:
class AverageMeter:
    def __init__(self): self.reset()
    def reset(self):    self.sum = self.count = 0.0
    def update(self, v, n=1): self.sum += v*n; self.count += n
    @property
    def avg(self): return self.sum / self.count if self.count else 0.0


def wall_iou(pred, target):
    p, t = pred.view(-1), target.view(-1)
    tp = (p==1) & (t==1); fp = (p==1) & (t==0); fn = (p==0) & (t==1)
    d  = tp.sum() + fp.sum() + fn.sum()
    return (tp.sum() / d).item() if d > 0 else 0.0


# ══════════════════════════════════════════════════════════════
# CheckpointManager
# ══════════════════════════════════════════════════════════════

class LoRACheckpointManager:
    """
    DINOv2+LoRA 专用检查点管理器

    文件名格式：
      {dataset_version}_bs{batch}_lr{lr}_ep{epoch:03d}_iou{score:.4f}.pth
    例如：
      combined_bs4_lr5e-5_ep015_iou0.7234.pth

    三类文件：
      Top-K 轮转 checkpoint  ← 每个 epoch 存，超出 save_top_k 自动淘汰最差
      best.pth               ← 覆盖写，始终是最新最优（固定名，方便快速找）
      best_ep{n}_iou{x}.pth  ← 最优快照，不被覆盖（方便回溯历史最优点）

    断点续训：
      load_for_resume() 恢复 model / optimizer / epoch
    """

    def __init__(self, cfg: PseudoLabelConfig,
                 save_top_k: int = 3,
                 monitor:    str = 'val_iou'):
        self.cfg        = cfg
        self.save_top_k = save_top_k
        self.monitor    = monitor
        self.scores     = []   # [(score, path), ...]
        self.ckpt_dir   = Path(cfg.checkpoint_dir)
        self.ckpt_dir.mkdir(parents=True, exist_ok=True)

        # 超参标签（嵌入所有文件名）
        lr_str    = f'{cfg.learning_rate:.0e}'.replace('-0', '-')
        self.tag  = f'{cfg.dataset_version}_bs{cfg.batch_size}_lr{lr_str}'

    def _make_path(self, epoch: int, score: float) -> Path:
        return self.ckpt_dir / f'{self.tag}_ep{epoch:03d}_iou{score:.4f}.pth'

    def _build_payload(self, model, optimizer, epoch: int,
                       metrics: dict, include_full: bool = True) -> dict:
        payload = {
            'epoch':           epoch,
            'dataset_version': self.cfg.dataset_version,
            'batch_size':      self.cfg.batch_size,
            'learning_rate':   self.cfg.learning_rate,
            'max_epochs':      self.cfg.max_epochs,
            'lora_r':          self.cfg.lora_r,
            'lora_alpha':      self.cfg.lora_alpha,
            'metrics':         metrics,
        }
        if include_full:
            payload['model_state']     = model.state_dict()
            payload['optimizer_state'] = optimizer.state_dict()
        return payload

    def save(self, model, optimizer, epoch: int, metrics: dict):
        """保存当前 epoch checkpoint，维护 Top-K"""
        score = metrics.get(self.monitor, 0.0)
        path  = self._make_path(epoch, score)
        torch.save(self._build_payload(model, optimizer, epoch, metrics), path)

        self.scores.append((score, path))
        self.scores.sort(key=lambda x: x[0], reverse=True)

        # 淘汰 Top-K 外最差的
        while len(self.scores) > self.save_top_k:
            _, old_path = self.scores.pop()
            if old_path.exists():
                old_path.unlink()
                logger.info(f'  [ckpt] 淘汰: {old_path.name}')

        rank = next(i+1 for i, (_, p) in enumerate(self.scores) if p == path)
        logger.info(f'  [ckpt] 保存: {path.name}  (Top-{self.save_top_k} 第{rank}位)')

    def save_best(self, model, optimizer, epoch: int,
                  metrics: dict) -> Path:
        """
        保存最优模型
          best.pth              ← 固定名，覆盖写
          best_ep{n}_iou{x}.pth ← 快照，不被覆盖
        同时保存 LoRA 增量（只有几 MB）
        """
        score     = metrics.get(self.monitor, 0.0)
        payload   = self._build_payload(model, optimizer, epoch, metrics)

        best_path = self.ckpt_dir / f'{self.tag}_best.pth'
        snap_path = self.ckpt_dir / f'{self.tag}_best_ep{epoch:03d}_iou{score:.4f}.pth'

        torch.save(payload, best_path)   # 覆盖写
        torch.save(payload, snap_path)   # 快照，不被覆盖

        # LoRA 增量单独保存（几 MB，方便分发）
        lora_path = self.ckpt_dir / f'{self.tag}_best_lora_only.pth'
        save_lora_weights(model.encoder, str(lora_path))

        logger.info(f'  [best] {best_path.name}  iou={score:.4f}  ep{epoch}')
        logger.info(f'  [snap] {snap_path.name}')
        logger.info(f'  [lora] {lora_path.name}')
        return best_path

    def load_for_resume(self, resume_path: str,
                        model, optimizer) -> int:
        """
        从 checkpoint 恢复 model + optimizer
        返回 start_epoch（下一个 epoch 编号）
        """
        ckpt = torch.load(resume_path, map_location='cpu')
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])

        start_epoch = ckpt['epoch'] + 1
        saved_iou   = ckpt.get('metrics', {}).get(self.monitor, 0.0)

        logger.info(f'  [resume] 从 {Path(resume_path).name} 恢复')
        logger.info(f'  [resume] 已完成 epoch={ckpt["epoch"]}  val_iou={saved_iou:.4f}')
        logger.info(f'  [resume] 继续从 epoch={start_epoch} 训练')

        # 重建 scores 列表（扫描目录内已有 Top-K 文件）
        for p in sorted(self.ckpt_dir.glob(f'{self.tag}_ep*.pth')):
            try:
                c = torch.load(p, map_location='cpu')
                s = c.get('metrics', {}).get(self.monitor, 0.0)
                self.scores.append((s, p))
            except Exception:
                pass
        self.scores.sort(key=lambda x: x[0], reverse=True)
        self.scores = self.scores[:self.save_top_k]
        logger.info(f'  [resume] 扫描到 {len(self.scores)} 个已有 checkpoint')
        return start_epoch


# ══════════════════════════════════════════════════════════════
# train_one_version：完整训练 + checkpoint 管理
# ══════════════════════════════════════════════════════════════

def train_one_version(version: str, base_cfg: PseudoLabelConfig = None):
    """
    对指定 dataset_version 完成一次完整训练。

    方式 A（单版本）:  train_one_version('hq')
    方式 B（全版本）:  RUN_ALL_VERSIONS = True

    断点续训: base_cfg.resume_from = '/path/to/xxx.pth'
    """
    import dataclasses

    cfg = dataclasses.replace(base_cfg or CFG, dataset_version=version)

    logger.info('=' * 65)
    logger.info(f'开始训练  version={version}')
    logger.info(f'  train     : {cfg.train_files}')
    logger.info(f'  val       : {cfg.val_files}')
    logger.info(f'  ckpt_dir  : {cfg.checkpoint_dir}')
    logger.info(f'  run_name  : {cfg.mlflow_run_name}')
    logger.info(f'  resume    : {cfg.resume_from}')
    logger.info('=' * 65)

    # ── 数据 ──
    train_ds = FloorplanDataset(cfg, 'train')
    val_ds   = FloorplanDataset(cfg, 'val')
    if len(train_ds) == 0:
        logger.error(f'version={version} 训练集为空，检查 split 文件是否已生成')
        return None

    train_loader = DataLoader(train_ds, cfg.batch_size, shuffle=True,
                              num_workers=cfg.num_workers, pin_memory=True,
                              drop_last=True, collate_fn=collate_fn)
    val_loader   = DataLoader(val_ds, cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=True,
                              collate_fn=collate_fn)
    logger.info(f'train batches={len(train_loader)}  val batches={len(val_loader)}')

    # ── 模型 ──
    model     = DINOv2LoRAModel(cfg).to(DEVICE)
    criterion = MultiTaskLoss(cfg).to(DEVICE)

    # 分层学习率
    lora_params  = [p for n, p in model.named_parameters()
                    if p.requires_grad and 'lora_' in n]
    head_params  = [p for n, p in model.named_parameters()
                    if p.requires_grad and any(k in n for k in
                    ('seg_head', 'det_fpn', 'rpn', 'roi_heads'))]
    other_params = [p for n, p in model.named_parameters()
                    if p.requires_grad
                    and p not in set(lora_params + head_params)]

    optimizer = torch.optim.AdamW([
        {'params': lora_params,  'lr': cfg.learning_rate,       'name': 'lora',  'initial_lr': cfg.learning_rate},
        {'params': head_params,  'lr': cfg.learning_rate * 5.0, 'name': 'head',  'initial_lr': cfg.learning_rate * 5.0},
        {'params': other_params, 'lr': cfg.learning_rate,       'name': 'other', 'initial_lr': cfg.learning_rate},
    ], weight_decay=cfg.weight_decay)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max   = max(cfg.max_epochs - cfg.warmup_epochs, 1),
        eta_min = cfg.learning_rate * 0.01,
    )
    scaler = torch.amp.GradScaler('cuda', enabled=(cfg.use_amp and DEVICE == 'cuda'))

    # ── Checkpoint Manager ──
    ckpt_mgr = LoRACheckpointManager(cfg, save_top_k=3)

    # ── 断点续训 ──
    start_epoch = 1
    best_iou    = 0.0
    if cfg.resume_from and os.path.exists(cfg.resume_from):
        start_epoch = ckpt_mgr.load_for_resume(cfg.resume_from, model, optimizer)
    elif cfg.resume_from:
        logger.warning(f'resume_from 路径不存在，从头训练: {cfg.resume_from}')

    # ── MLflow ──
    mlflow.set_tracking_uri(f'file://{MLFLOW_DIR}')
    mlflow.set_experiment(cfg.mlflow_experiment)

    best_path = None

    with mlflow.start_run(run_name=cfg.mlflow_run_name) as run:
        lr_str = f'{cfg.learning_rate:.0e}'.replace('-0', '-')
        mlflow.log_params({
            'dataset_version': version,
            'backbone':        cfg.dinov2_model,
            'lora_r':          cfg.lora_r,
            'lora_alpha':      cfg.lora_alpha,
            'lr':              cfg.learning_rate,
            'max_epochs':      cfg.max_epochs,
            'batch_size':      cfg.batch_size,
            'warmup_epochs':   cfg.warmup_epochs,
            'train_n':         len(train_ds),
            'val_n':           len(val_ds),
            'resume_from':     str(cfg.resume_from),
            'start_epoch':     start_epoch,
        })
        logger.info(f'MLflow run_id={run.info.run_id}')

        for epoch in range(start_epoch, cfg.max_epochs + 1):

            # ── Warmup ──
            if epoch <= cfg.warmup_epochs:
                scale = epoch / cfg.warmup_epochs
                for pg in optimizer.param_groups:
                    pg['lr'] = pg['initial_lr'] * scale

            # ── Train ──
            model.train(); criterion.train()
            meters = {k: AverageMeter() for k in ('total', 'seg', 'det')}
            t0 = time.time()

            for batch in tqdm(train_loader,
                              desc=f'[{version}] Ep{epoch:03d} train',
                              leave=False):
                images  = batch['image'].to(DEVICE)
                masks   = batch['mask'].to(DEVICE)
                targets = [{'boxes': b.to(DEVICE), 'labels': l.to(DEVICE)}
                           for b, l in zip(batch['boxes'], batch['labels'])]

                optimizer.zero_grad()
                with torch.amp.autocast(device_type=DEVICE,
                                        enabled=(cfg.use_amp and DEVICE == 'cuda')):
                    out = model(images, targets)
                    ld  = criterion(out['seg_logits'], masks, out['det_losses'])

                scaler.scale(ld['loss_total']).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], cfg.grad_clip)
                scaler.step(optimizer); scaler.update()

                n = images.size(0)
                meters['total'].update(ld['loss_total'].item(), n)
                meters['seg'].update(ld['loss_seg'], n)
                meters['det'].update(ld['loss_det'], n)

            # ── Val ──
            model.eval(); iou_m = AverageMeter()
            with torch.no_grad():
                for batch in tqdm(val_loader,
                                  desc=f'[{version}] Ep{epoch:03d} val  ',
                                  leave=False):
                    images = batch['image'].to(DEVICE)
                    masks  = batch['mask'].to(DEVICE)
                    out    = model(images)
                    preds  = out['seg_logits'].argmax(dim=1)
                    iou_m.update(wall_iou(preds.cpu(), masks.cpu()), images.size(0))

            if epoch > cfg.warmup_epochs:
                scheduler.step()
            lr = optimizer.param_groups[0]['lr']

            metrics = {
                'val_iou':    iou_m.avg,
                'train_loss': meters['total'].avg,
                'train_seg':  meters['seg'].avg,
                'train_det':  meters['det'].avg,
            }

            # ── MLflow ──
            mlflow.log_metrics({**metrics, 'lr': lr}, step=epoch)

            elapsed = time.time() - t0
            logger.info(
                f'[{version}] Ep{epoch:03d}  '
                f'loss={meters["total"].avg:.4f}  '
                f'seg={meters["seg"].avg:.4f}  '
                f'det={meters["det"].avg:.4f}  '
                f'val_iou={iou_m.avg:.4f}  '
                f'lr={lr:.2e}  {elapsed:.0f}s'
            )

            # ── 每 epoch 保存 Top-K checkpoint ──
            ckpt_mgr.save(model, optimizer, epoch, metrics)

            # ── 保存最优 ──
            if iou_m.avg > best_iou:
                best_iou  = iou_m.avg
                best_path = ckpt_mgr.save_best(model, optimizer, epoch, metrics)
                mlflow.log_metric('best_val_iou', best_iou, step=epoch)
                mlflow.log_param('best_ckpt_name', best_path.name)

        logger.info(f'[{version}] 训练完成  best_val_iou={best_iou:.4f}')

        # ── 训练结束：保存结果 JSON + GCS 上传 + 版本索引 ──
        lr_str  = f'{cfg.learning_rate:.0e}'.replace('-0', '-')
        run_tag = (f'{version}'
                   f'_ep{cfg.max_epochs}'
                   f'_bs{cfg.batch_size}'
                   f'_lr{lr_str}')

        # 结果 JSON（文件名含版本 + 超参）
        results = {
            'model':           'DINOv2_ViT-L_LoRA',
            'dataset_version': version,
            'train_files':     cfg.train_files,
            'best_val_iou':    best_iou,
            'epochs_trained':  cfg.max_epochs,
            'start_epoch':     start_epoch,
            'batch_size':      cfg.batch_size,
            'learning_rate':   cfg.learning_rate,
            'lora_r':          cfg.lora_r,
            'lora_alpha':      cfg.lora_alpha,
            'best_ckpt_name':  best_path.name if best_path else None,
            'training_date':   time.strftime('%Y-%m-%d %H:%M:%S'),
            'mlflow_run_id':   run.info.run_id,
            'gcs_dir':         f'gs://{cfg.gcs_bucket}/{cfg.gcs_model_dir}/',
        }
        results_filename = f'{run_tag}_results.json'
        results_path     = os.path.join(cfg.checkpoint_dir, results_filename)
        with open(results_path, 'w') as f_out:
            json.dump(results, f_out, indent=2)
        logger.info(f'结果 JSON: {results_path}')

        # MLflow：补充 artifact + 版本参数
        if best_path:
            mlflow.log_artifact(str(best_path), 'model')
            mlflow.log_artifact(results_path,   'results')
            # LoRA 增量单独上传（几 MB）
            lora_path = ckpt_mgr.ckpt_dir / f'{ckpt_mgr.tag}_best_lora_only.pth'
            if lora_path.exists():
                mlflow.log_artifact(str(lora_path), 'model')
            mlflow.log_params({
                'results_file': results_filename,
                'gcs_dir':      results['gcs_dir'],
            })

        # GCS 上传：按 dataset_version/run_tag 独立子目录
        GSUTIL  = '/root/google-cloud-sdk/bin/gsutil'
        gcs_dir = f'gs://{cfg.gcs_bucket}/{cfg.gcs_model_dir}/'
        logger.info(f'GCS 上传目标: {gcs_dir}')

        upload_list = []
        if best_path and best_path.exists():
            upload_list.append((str(best_path), f'{gcs_dir}{best_path.name}'))
            # LoRA 增量
            if lora_path.exists():
                upload_list.append((str(lora_path), f'{gcs_dir}{lora_path.name}'))
        upload_list.append((results_path, f'{gcs_dir}{results_filename}'))

        for src_file, dst in upload_list:
            ret = subprocess.run(
                f'{GSUTIL} cp {src_file} {dst}',
                shell=True, capture_output=True, text=True,
                env={**os.environ,
                     'PATH': f'/root/google-cloud-sdk/bin:{os.environ["PATH"]}'}
            )
            symbol = '✓' if ret.returncode == 0 else '✗'
            logger.info(f'  {symbol} {dst.split("/")[-1]}')
            if ret.returncode != 0 and ret.stderr:
                logger.warning(f'    {ret.stderr.strip()[:100]}')

        # 本地版本索引：追加写入，不覆盖已有记录
        # 文件: CKPT_DIR/version_index.json（所有版本共享）
        index_path = os.path.join(CKPT_DIR, 'version_index.json')
        existing   = {}
        if os.path.exists(index_path):
            with open(index_path) as f_idx:
                existing = json.load(f_idx)
        existing[run_tag] = {
            'dataset_version': version,
            'gcs_dir':         gcs_dir,
            'best_ckpt':       best_path.name if best_path else None,
            'best_val_iou':    round(best_iou, 4),
            'mlflow_run_id':   run.info.run_id,
            'training_date':   results['training_date'],
        }
        with open(index_path, 'w') as f_idx:
            json.dump(existing, f_idx, indent=2)
        logger.info(f'版本索引更新: {index_path}  (共 {len(existing)} 条记录)')

    return {
        'version':      version,
        'best_val_iou': best_iou,
        'best_ckpt':    str(best_path) if best_path else None,
        'ckpt_dir':     str(ckpt_mgr.ckpt_dir),
        'gcs_dir':      gcs_dir if best_path else None,
        'run_tag':      run_tag,
    }


# ══════════════════════════════════════════════════════════════
# 训练入口
# ══════════════════════════════════════════════════════════════

RUN_TRAINING     = True   # ← 方式 A：只训练 CFG.dataset_version
RUN_ALL_VERSIONS = False   # ← 方式 B：顺序跑 hq / hq_arch / combined

if RUN_ALL_VERSIONS:
    all_results = {}
    for v in ('hq', 'hq_arch', 'combined'):
        r = train_one_version(v)
        if r:
            all_results[v] = r

    print('\n三版本训练结果汇总：')
    print(f'{"版本":<12} {"val_iou":>9}  {"checkpoint"}')
    print('─' * 70)
    for v, r in all_results.items():
        print(f'{v:<12} {r["best_val_iou"]:>9.4f}  {Path(r["best_ckpt"]).name}')
        print(f'{"":<12} {"":>9}    gcs: {r["gcs_dir"]}')

elif RUN_TRAINING:
    r = train_one_version(CFG.dataset_version)
    if r:
        print(f'\n训练完成  version={r["version"]}  best_val_iou={r["best_val_iou"]:.4f}')
        print(f'checkpoint: {r["best_ckpt"]}')

else:
    print('跳过训练')
    print('  方式 A: CFG.dataset_version = "hq"，RUN_TRAINING = True')
    print('  方式 B: RUN_ALL_VERSIONS = True  顺序跑三个版本')
    print('  断点续训: CFG.resume_from = "/path/to/xxx.pth"')


05:33:01 [INFO] =================================================================
05:33:01 [INFO] 开始训练  version=combined
05:33:01 [INFO]   train     : ['train_hq.txt', 'train_hq_arch.txt']
05:33:01 [INFO]   val       : ['val_hq.txt', 'val_hq_arch.txt']
05:33:01 [INFO]   ckpt_dir  : /workspace/production_3d/checkpoints_dinov2_lora/combined
05:33:01 [INFO]   run_name  : combined_ep50_bs4_lr5e-5_0512_0533
05:33:01 [INFO]   resume    : None
05:33:01 [INFO] =================================================================
05:33:01 [INFO] [train] version=combined  files=['train_hq.txt', 'train_hq_arch.txt']  →  3778 个样本
05:33:01 [INFO] [val] version=combined  files=['val_hq.txt', 'val_hq_arch.txt']  →  472 个样本
05:33:01 [INFO] train batches=944  val batches=118
05:33:03 [INFO] Loading pretrained weights from Hugging Face hub (timm/vit_large_patch14_dinov2.lvd142m)
05:33:03 [INFO] HTTP Request: HEAD https://huggingface.co/timm/vit_large_patch14_dinov2.lvd142m/resolve/main/model.safetensors "HT

[combined] Ep001 train:   0%|          | 0/944 [00:00<?, ?it/s]

libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 'CMYK': invalid ICC profile color space
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libp

[combined] Ep001 val  :   0%|          | 0/118 [00:00<?, ?it/s]

libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
05:48:02 [INFO] [combined] Ep001  loss=0.9208  seg=0.6148  det=0.3060  val_iou=0.2130  lr=1.67e-05  898s
05:48:03 [INFO]   [ckpt] 保存: combin

...

[combined] Ep008 train:   0%|          | 0/944 [00:00<?, ?it/s]

libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 'CMYK': invalid ICC profile color space
libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warn

[combined] Ep008 val  :   0%|          | 0/118 [00:00<?, ?it/s]

libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
07:29:03 [INFO] [combined] Ep008  loss=0.7263  seg=0.5512  det=0.1751  val_iou=0.2364  lr=4.86e-05  874s
07:29:04 [INFO]   [ckpt] 淘汰: combin

StopIteration: 

## Cell 6 · Step 2：SAM2 精化初始 Mask

In [8]:
# ══════════════════════════════════════════════════════════════
# DINOv2+LoRA 对公司图纸推理 → 初始 mask
# SAM2 Point Prompt 精化 → 边界更清晰的 refined mask
#
# 点采样策略：
#   正点：在 pred_mask=1 的区域内，用侵蚀后的中心区域采样
#          （避免采到噪声边界）
#   负点：在 pred_mask=0 的区域内随机采样
# ══════════════════════════════════════════════════════════════

def sample_points_from_mask(
    mask:       np.ndarray,
    n_pos:      int = 5,
    n_neg:      int = 3,
    erode_px:   int = 8,     # 正点侵蚀像素，避免采到边界附近
) -> Tuple[np.ndarray, np.ndarray]:
    """
    从 binary mask 采样正负点
    返回: points (N,2) [x,y],  labels (N,) 1=正 0=负
    """
    h, w = mask.shape
    points, labels = [], []

    # ── 正点：侵蚀后的前景区域 ──
    kernel   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (erode_px*2+1, erode_px*2+1))
    eroded   = cv2.erode(mask.astype(np.uint8), kernel, iterations=1)
    pos_yx   = np.argwhere(eroded > 0)   # (N, 2) [row, col]
    if len(pos_yx) >= n_pos:
        chosen = pos_yx[np.random.choice(len(pos_yx), n_pos, replace=False)]
        for r, c in chosen:
            points.append([c, r]); labels.append(1)   # SAM2 用 [x,y]
    elif len(pos_yx) > 0:
        chosen = pos_yx[np.random.choice(len(pos_yx), min(n_pos, len(pos_yx)), replace=True)]
        for r, c in chosen:
            points.append([c, r]); labels.append(1)

    # ── 负点：背景区域 ──
    neg_yx = np.argwhere(mask == 0)
    if len(neg_yx) >= n_neg:
        chosen = neg_yx[np.random.choice(len(neg_yx), n_neg, replace=False)]
        for r, c in chosen:
            points.append([c, r]); labels.append(0)

    if not points:
        return np.zeros((0,2), dtype=np.float32), np.zeros(0, dtype=np.int32)
    return np.array(points, dtype=np.float32), np.array(labels, dtype=np.int32)


def refine_mask_with_sam2(
    image_rgb:    np.ndarray,
    initial_mask: np.ndarray,
    sam2_predictor,
    cfg:          PseudoLabelConfig,
) -> np.ndarray:
    """
    用 SAM2 精化初始 mask

    流程：
      1. set_image → SAM2 编码图像特征
      2. 从 initial_mask 采样正负点
      3. predict → 多个 mask 候选
      4. 选 score 最高且与 initial_mask IoU 最大的
    """
    if not HAS_SAM2:
        logger.warning('SAM2 未安装，返回初始 mask')
        return initial_mask

    sam2_predictor.set_image(image_rgb)

    points, point_labels = sample_points_from_mask(
        initial_mask,
        n_pos     = cfg.sam2_n_pos_points,
        n_neg     = cfg.sam2_n_neg_points,
    )
    if len(points) == 0:
        return initial_mask

    masks, scores, _ = sam2_predictor.predict(
        point_coords = points,
        point_labels = point_labels,
        multimask_output = True,   # 输出多个候选
    )
    # masks: (N_masks, H, W)  scores: (N_masks,)

    # 选与初始 mask IoU 最大的候选（语义一致性优先于 SAM2 自身置信度）
    best_idx, best_iou_val = 0, -1.0
    init_bool = initial_mask.astype(bool)
    for i, (m, s) in enumerate(zip(masks, scores)):
        if s < cfg.sam2_score_thresh:
            continue
        m_bool  = m.astype(bool)
        inter   = (m_bool & init_bool).sum()
        union   = (m_bool | init_bool).sum()
        iou_val = inter / (union + 1e-8)
        if iou_val > best_iou_val:
            best_iou_val = iou_val
            best_idx     = i

    refined = masks[best_idx].astype(np.uint8)
    logger.debug(f'SAM2 精化  init_pixels={initial_mask.sum()}  '
                 f'refined_pixels={refined.sum()}  iou_with_init={best_iou_val:.3f}')
    return refined


def load_sam2_predictor(cfg: PseudoLabelConfig):
    """加载 SAM2 ImagePredictor，懒加载避免 OOM"""
    if not HAS_SAM2:
        return None
    if not os.path.exists(cfg.sam2_ckpt):
        logger.error(f'SAM2 权重不存在: {cfg.sam2_ckpt}')
        logger.error('下载: wget https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt')
        return None
    sam2_model = build_sam2(cfg.sam2_cfg, cfg.sam2_ckpt, device=DEVICE)
    predictor  = SAM2ImagePredictor(sam2_model)
    logger.info('✓ SAM2 ImagePredictor 加载完成')
    return predictor


print('✓ SAM2 精化函数定义完成')
print('  sample_points_from_mask : 正点从侵蚀后的前景采样，负点从背景采样')
print('  refine_mask_with_sam2   : Point Prompt → 多候选 → IoU 最大选择')

✓ SAM2 精化函数定义完成
  sample_points_from_mask : 正点从侵蚀后的前景采样，负点从背景采样
  refine_mask_with_sam2   : Point Prompt → 多候选 → IoU 最大选择


## Cell 7 · Step 3：VLM 语义补全

In [9]:
import base64, io as _io
from PIL import Image as PILImage


VLM_SYSTEM_PROMPT = """你是一位专业的建筑图纸识别工程师。
你将收到一张平面图图片和一张对应的墙体分割 mask。
你的任务是识别图中所有的门和窗，并输出结构化的 JSON 数据。
输出必须是合法的 JSON，不包含任何 Markdown 标记或解释文字。"""

VLM_USER_PROMPT = """请仔细观察这张平面图（第一张图）和对应的墙体 mask（第二张图）。

识别所有门和窗，对每个开口输出：
  - type: "door" 或 "window"
  - bbox: [x1, y1, x2, y2]（像素坐标，左上角到右下角）
  - wall_side: "north"/"south"/"east"/"west"（开口所在墙面朝向，不确定填 "unknown"）
  - estimated_width_m: 估算宽度（米），门通常 0.8~1.2m，窗通常 0.6~2.4m
  - confidence: 0.0~1.0（你对这个识别的置信度）

同时输出整体信息：
  - n_rooms: 估计的房间数量
  - floor_area_m2: 估计的建筑面积（平方米），不确定填 null

返回格式（严格 JSON）：
{
  "openings": [
    {"type": "door", "bbox": [x1,y1,x2,y2], "wall_side": "north",
     "estimated_width_m": 0.9, "confidence": 0.95},
    ...
  ],
  "n_rooms": 3,
  "floor_area_m2": 85.0
}"""


def _img_to_b64(image_rgb: np.ndarray) -> str:
    pil = PILImage.fromarray(image_rgb.astype(np.uint8))
    buf = _io.BytesIO()
    pil.save(buf, format='PNG')
    return base64.standard_b64encode(buf.getvalue()).decode()


def _parse_vlm_json(text: str) -> dict:
    text = text.strip()
    if text.startswith('```'):
        lines = text.split('\n')
        text  = '\n'.join(lines[1:-1] if lines[-1].strip()=='```' else lines[1:])
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        logger.warning(f'VLM JSON 解析失败: {text[:200]}')
        return {'openings': [], 'n_rooms': None, 'floor_area_m2': None,
                '_parse_error': True}


def vlm_semantic_completion(
    image_rgb:    np.ndarray,
    refined_mask: np.ndarray,
    cfg:          PseudoLabelConfig,
    client,
) -> dict:
    """
    Step 3: VLM 语义补全
    输入：原图 + refined_mask
    输出：openings 的语义信息 JSON

    mask 可视化策略：
      把 refined_mask 叠加到原图上（绿色半透明），让 VLM 同时看到
      图像纹理（识别门窗符号）和墙体位置（定位开口所在墙）
    """
    if not HAS_ANTHROPIC:
        logger.error('pip install anthropic')
        return {'openings': [], '_error': 'anthropic not installed'}

    # 把 mask 叠加到原图，生成可视化图
    vis = image_rgb.copy()
    vis[refined_mask == 1] = (
        vis[refined_mask == 1] * 0.6 + np.array([0, 200, 0]) * 0.4
    ).astype(np.uint8)

    img_b64  = _img_to_b64(image_rgb)
    mask_b64 = _img_to_b64(vis)

    response = client.messages.create(
        model      = cfg.vlm_model,
        max_tokens = cfg.vlm_max_tokens,
        system     = VLM_SYSTEM_PROMPT,
        messages   = [{
            'role': 'user',
            'content': [
                {'type': 'image', 'source': {'type': 'base64',
                  'media_type': 'image/png', 'data': img_b64}},
                {'type': 'image', 'source': {'type': 'base64',
                  'media_type': 'image/png', 'data': mask_b64}},
                {'type': 'text', 'text': VLM_USER_PROMPT},
            ]
        }]
    )

    result = _parse_vlm_json(response.content[0].text)
    result['_tokens'] = {
        'input':  response.usage.input_tokens,
        'output': response.usage.output_tokens,
    }
    return result


print('✓ VLM 语义补全函数定义完成')
print('  输入: 原图 + refined_mask（绿色叠加可视化）')
print('  输出: openings JSON（type/bbox/wall_side/width/confidence）')

✓ VLM 语义补全函数定义完成
  输入: 原图 + refined_mask（绿色叠加可视化）
  输出: openings JSON（type/bbox/wall_side/width/confidence）


## Cell 8 · Step 4：SVG 脚本生成

In [10]:
# ══════════════════════════════════════════════════════════════
# SVG 生成
# 输入: wall_boxes (Shrinking 矢量化结果) + openings (VLM 输出)
# 输出: CubiCasa 兼容格式的 SVG 文件
#
# CubiCasa SVG 关键约定：
#   <g id="Floor0"> 包含所有楼层元素
#   <rect> 表示墙体
#   <path> 的 class="Wall" 表示墙
#   门用 class="Door" 的 <use> 或 <rect>
#   窗用 class="Window" 的 <rect>
# ══════════════════════════════════════════════════════════════

def generate_svg(
    image_wh:   Tuple[int, int],
    wall_boxes: List[Tuple],
    openings:   List[dict],
    cfg:        PseudoLabelConfig,
    output_path: str,
) -> str:
    """
    生成 CubiCasa 兼容的 SVG 文件

    openings 格式（VLM 输出合并矢量化结果后）：
    [{'type': 'door'/'window',
      'bbox': [x1,y1,x2,y2],
      'wall_side': 'north'/...}, ...]
    """
    W, H = image_wh
    lines = []

    # ── SVG 头部 ──
    lines.append(f'<?xml version="1.0" encoding="utf-8"?>')
    lines.append(f'<svg xmlns="http://www.w3.org/2000/svg" '
                 f'xmlns:xlink="http://www.w3.org/1999/xlink" '
                 f'width="{W}" height="{H}" viewBox="0 0 {W} {H}">')

    # ── 楼层组 ──
    lines.append('  <g id="Floor0">')

    # ── 墙体 ──
    lines.append('    <!-- Walls -->')
    for i, b in enumerate(wall_boxes):
        x1, y1, x2, y2 = [int(v) for v in b]
        bw = max(x2 - x1, 1)
        bh = max(y2 - y1, 1)
        lines.append(
            f'    <rect id="wall_{i}" class="Wall" '
            f'x="{x1}" y="{y1}" width="{bw}" height="{bh}" '
            f'fill="#333333" stroke="none"/>'
        )

    # ── 门 ──
    lines.append('    <!-- Doors -->')
    door_count = 0
    for op in openings:
        if op.get('type') != 'door': continue
        b  = op['bbox']
        x1,y1,x2,y2 = [int(v) for v in b]
        bw = max(x2-x1, 1); bh = max(y2-y1, 1)
        conf = op.get('confidence', 1.0)
        lines.append(
            f'    <rect id="door_{door_count}" class="Door" '
            f'x="{x1}" y="{y1}" width="{bw}" height="{bh}" '
            f'fill="#8B4513" stroke="none" '
            f'data-confidence="{conf:.2f}" '
            f'data-wall-side="{op.get("wall_side","unknown")}"/>'
        )
        door_count += 1

    # ── 窗 ──
    lines.append('    <!-- Windows -->')
    win_count = 0
    for op in openings:
        if op.get('type') != 'window': continue
        b  = op['bbox']
        x1,y1,x2,y2 = [int(v) for v in b]
        bw = max(x2-x1, 1); bh = max(y2-y1, 1)
        conf = op.get('confidence', 1.0)
        lines.append(
            f'    <rect id="window_{win_count}" class="Window" '
            f'x="{x1}" y="{y1}" width="{bw}" height="{bh}" '
            f'fill="#87CEEB" stroke="#4169E1" stroke-width="1" '
            f'data-confidence="{conf:.2f}" '
            f'data-wall-side="{op.get("wall_side","unknown")}"/>'
        )
        win_count += 1

    # ── 元数据注释 ──
    lines.append(f'    <!-- Meta: walls={len(wall_boxes)} doors={door_count} windows={win_count} '
                 f'generated={time.strftime("%Y-%m-%dT%H:%M:%SZ")} -->')

    lines.append('  </g>')
    lines.append('</svg>')

    svg_content = '\n'.join(lines)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(svg_content)

    logger.info(f'SVG 生成完成: {output_path}  '
                f'walls={len(wall_boxes)}  doors={door_count}  windows={win_count}')
    return svg_content


print('✓ SVG 生成函数定义完成')
print('  输出格式: CubiCasa 兼容 SVG')
print('  Wall → <rect class="Wall">')
print('  Door → <rect class="Door" data-confidence data-wall-side>')
print('  Window → <rect class="Window" data-confidence data-wall-side>')

✓ SVG 生成函数定义完成
  输出格式: CubiCasa 兼容 SVG
  Wall → <rect class="Wall">
  Door → <rect class="Door" data-confidence data-wall-side>
  Window → <rect class="Window" data-confidence data-wall-side>


## Cell 9 · 完整四步流水线（单张图片）

In [11]:
from torchvision.ops import nms as tv_nms


def run_pseudo_label_pipeline(
    image_path:    str,
    cfg:           PseudoLabelConfig,
    dinov2_model,
    sam2_predictor,
    vlm_client,
    dry_run_vlm:   bool = True,   # True 时跳过 VLM，用检测头结果代替
) -> dict:
    """
    单张公司图纸的完整四步伪标注生成流水线

    返回:
    {
        'image_path', 'svg_path',
        'wall_boxes', 'openings',
        'metrics': {n_walls, n_doors, n_windows, elapsed}
    }
    """
    from vector_logic import vectorize_wall_mask
    name   = Path(image_path).stem
    t0     = time.time()
    tf     = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(cfg.norm_mean, cfg.norm_std),
    ])

    logger.info(f'开始处理: {name}')

    # ── 读图 ──
    img_bgr = cv2.imread(image_path)
    assert img_bgr is not None, f'图片不存在: {image_path}'
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    H, W    = img_rgb.shape[:2]

    # ════════════════════════════════════════
    # Step 1: DINOv2+LoRA 推理 → 初始 mask
    # ════════════════════════════════════════
    ts     = cfg.tile_size
    stride = ts - cfg.tile_overlap
    ys = list(range(0, max(H-ts+1,1), stride)); xs = list(range(0, max(W-ts+1,1), stride))
    if not ys or ys[-1]+ts < H: ys.append(max(H-ts,0))
    if not xs or xs[-1]+ts < W: xs.append(max(W-ts,0))

    wall_prob = np.zeros((H,W), dtype=np.float32)
    wall_cnt  = np.zeros((H,W), dtype=np.float32)
    all_boxes, all_scores, all_labels = [], [], []

    dinov2_model.eval()
    for ty in ys:
        for tx in xs:
            tile   = img_rgb[ty:ty+ts, tx:tx+ts].copy()
            th, tw = tile.shape[:2]
            if th<ts or tw<ts:
                tile = cv2.copyMakeBorder(tile,0,ts-th,0,ts-tw,cv2.BORDER_REFLECT_101)
            tile   = cv2.resize(tile,(ts,ts))
            tensor = tf(tile).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                out  = dinov2_model(tensor)
            prob = torch.softmax(out['seg_logits'],dim=1)[0,1].cpu().numpy()
            prob = cv2.resize(prob,(tw,th))
            wall_prob[ty:ty+th,tx:tx+tw] += prob[:th,:tw]
            wall_cnt[ty:ty+th,  tx:tx+tw] += 1
            for det in out['det_outputs']:
                bxs=det['boxes'].cpu().numpy(); scrs=det['scores'].cpu().numpy(); lbls=det['labels'].cpu().numpy()
                sx,sy = tw/ts, th/ts
                for b,s,l in zip(bxs,scrs,lbls):
                    if s>=0.5:
                        all_boxes.append([b[0]*sx+tx,b[1]*sy+ty,b[2]*sx+tx,b[3]*sy+ty])
                        all_scores.append(float(s)); all_labels.append(int(l))

    wall_prob   /= np.maximum(wall_cnt, 1)
    initial_mask = (wall_prob > 0.5).astype(np.uint8)

    if all_boxes:
        keep = tv_nms(torch.tensor(all_boxes), torch.tensor(all_scores), 0.5)
        det_boxes  = np.array(all_boxes)[keep.numpy()]
        det_labels = np.array(all_labels)[keep.numpy()]
    else:
        det_boxes  = np.zeros((0,4),dtype=np.float32)
        det_labels = np.zeros(0,dtype=np.int64)

    logger.info(f'Step1 完成  wall%={initial_mask.mean()*100:.1f}%  det={len(det_boxes)}')

    # ════════════════════════════════════════
    # Step 2: SAM2 精化
    # ════════════════════════════════════════
    if sam2_predictor is not None:
        refined_mask = refine_mask_with_sam2(img_rgb, initial_mask, sam2_predictor, cfg)
        logger.info(f'Step2 完成  refined_pixels={refined_mask.sum()}')
    else:
        refined_mask = initial_mask
        logger.info('Step2 跳过（SAM2 未安装）')

    # ════════════════════════════════════════
    # Step 3: VLM 语义补全
    # ════════════════════════════════════════
    if dry_run_vlm or vlm_client is None:
        # dry_run: 用检测头的结果当 openings，跳过 VLM API 调用
        vlm_meta = {'openings': [], '_dry_run': True}
        for b, l in zip(det_boxes, det_labels):
            vlm_meta['openings'].append({
                'type':             'door' if l==1 else 'window',
                'bbox':             [int(v) for v in b],
                'wall_side':        'unknown',
                'estimated_width_m': None,
                'confidence':       0.9,
            })
        logger.info(f'Step3 dry_run  openings={len(vlm_meta["openings"])}')
    else:
        vlm_meta = vlm_semantic_completion(img_rgb, refined_mask, cfg, vlm_client)
        logger.info(f'Step3 完成  openings={len(vlm_meta["openings"])}  '
                    f'tokens={vlm_meta.get("_tokens",{})}')

    # ════════════════════════════════════════
    # Step 4: 矢量化 + SVG 生成
    # ════════════════════════════════════════
    from postprocess_config import VectorizationConfig
    vect_cfg   = VectorizationConfig(iou_threshold=cfg.shrink_iou_thresh,
                                     min_segment_area=cfg.min_segment_area)
    wall_boxes = vectorize_wall_mask(refined_mask, vect_cfg)
    logger.info(f'Step4 矢量化  wall_boxes={len(wall_boxes)}')

    # 合并 VLM openings
    svg_path = os.path.join(cfg.pseudo_out_dir, name, 'pseudo_label.svg')
    generate_svg(
        image_wh    = (W, H),
        wall_boxes  = wall_boxes,
        openings    = vlm_meta['openings'],
        cfg         = cfg,
        output_path = svg_path,
    )

    # 保存中间结果（方便调试和复查）
    out_dir = os.path.join(cfg.pseudo_out_dir, name)
    cv2.imwrite(os.path.join(out_dir, 'initial_mask.png'),   initial_mask*255)
    cv2.imwrite(os.path.join(out_dir, 'refined_mask.png'),   refined_mask*255)
    with open(os.path.join(out_dir, 'semantic_meta.json'), 'w') as f:
        json.dump(vlm_meta, f, indent=2, ensure_ascii=False)

    elapsed = round(time.time()-t0, 2)
    doors   = [o for o in vlm_meta['openings'] if o['type']=='door']
    windows = [o for o in vlm_meta['openings'] if o['type']=='window']
    logger.info(f'完成  {elapsed}s  walls={len(wall_boxes)}  doors={len(doors)}  windows={len(windows)}')

    return {
        'image_path': image_path,
        'svg_path':   svg_path,
        'wall_boxes': wall_boxes,
        'openings':   vlm_meta['openings'],
        'metrics':    {'n_walls':len(wall_boxes), 'n_doors':len(doors),
                       'n_windows':len(windows), 'elapsed':elapsed},
    }


print('✓ 完整四步流水线定义完成')
print('  run_pseudo_label_pipeline(image_path, cfg, model, sam2, vlm, dry_run_vlm=True)')

✓ 完整四步流水线定义完成
  run_pseudo_label_pipeline(image_path, cfg, model, sam2, vlm, dry_run_vlm=True)


## Cell 10 · 测试：用一张 CubiCasa 图验证全流程

In [13]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

RUN_TEST = True   # ← 训练完成后改为 True

if RUN_TEST:
    # ── 加载训练好的 DINOv2+LoRA 模型 ──
    BEST_CKPT = os.path.join(CFG.checkpoint_dir, 'best_dinov2_lora.pth')
    assert os.path.exists(BEST_CKPT), f'先训练模型: {BEST_CKPT}'

    ckpt  = torch.load(BEST_CKPT, map_location=DEVICE)
    model = DINOv2LoRAModel(CFG).to(DEVICE)
    model.load_state_dict(ckpt['model_state'])
    logger.info(f'模型加载完成  epoch={ckpt["epoch"]}  val_iou={ckpt["val_iou"]:.4f}')

    # ── 加载 SAM2（可选）──
    sam2_pred = load_sam2_predictor(CFG)

    # ── VLM 客户端（dry_run=True 时不需要）──
    vlm_client = anthropic.Anthropic() if HAS_ANTHROPIC else None

    # ── 选一张 CubiCasa val 图做测试 ──
    from numpy import genfromtxt
    val_folders = genfromtxt(os.path.join(CFG.data_folder, 'val.txt'), dtype='str')
    test_folder = val_folders[0].strip('/')
    test_image  = os.path.join(CFG.data_folder, test_folder, 'F1_scaled.png')
    logger.info(f'测试图片: {test_image}')

    # ── 运行流水线 ──
    result = run_pseudo_label_pipeline(
        image_path    = test_image,
        cfg           = CFG,
        dinov2_model  = model,
        sam2_predictor= sam2_pred,
        vlm_client    = vlm_client,
        dry_run_vlm   = True,   # ← False 时调用真实 VLM
    )

    # ── 可视化四步结果 ──
    out_dir  = os.path.join(CFG.pseudo_out_dir, Path(test_image).stem)
    img_rgb  = cv2.cvtColor(cv2.imread(test_image), cv2.COLOR_BGR2RGB)
    init_m   = cv2.imread(os.path.join(out_dir, 'initial_mask.png'),  cv2.IMREAD_GRAYSCALE)
    refined_m= cv2.imread(os.path.join(out_dir, 'refined_mask.png'),  cv2.IMREAD_GRAYSCALE)

    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    axes[0].imshow(img_rgb);    axes[0].set_title('原始图片');              axes[0].axis('off')
    axes[1].imshow(init_m, cmap='gray'); axes[1].set_title('Step1: DINOv2+LoRA 初始 mask'); axes[1].axis('off')
    axes[2].imshow(refined_m, cmap='gray'); axes[2].set_title('Step2: SAM2 精化 mask'); axes[2].axis('off')

    # Step4: SVG 结果叠加可视化
    vis = img_rgb.copy()
    if refined_m is not None:
        vis[refined_m>128] = (vis[refined_m>128]*0.5 + np.array([0,200,0])*0.5).astype(np.uint8)
    for b in result['wall_boxes']:
        cv2.rectangle(vis,(int(b[0]),int(b[1])),(int(b[2]),int(b[3])),(0,255,80),2)
    for op in result['openings']:
        b = op['bbox']; c = (0,0,255) if op['type']=='door' else (255,100,0)
        cv2.rectangle(vis,(int(b[0]),int(b[1])),(int(b[2]),int(b[3])),c,2)
    axes[3].imshow(vis)
    axes[3].set_title(f'Step4: SVG  walls={result["metrics"]["n_walls"]}  '
                      f'doors={result["metrics"]["n_doors"]}  '
                      f'windows={result["metrics"]["n_windows"]}')
    axes[3].axis('off')

    legend = [
        mpatches.Patch(color='#00C850', label='Wall (mask)'),
        mpatches.Patch(color='#0000FF', label='Door'),
        mpatches.Patch(color='#FF6400', label='Window'),
    ]
    axes[3].legend(handles=legend, loc='lower right', fontsize=8)
    plt.suptitle(f'伪标注生成结果  {Path(test_image).stem}', fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'pipeline_result.png'), dpi=100)
    plt.show()

    print(f'\nSVG 文件: {result["svg_path"]}')
    print(f'耗时: {result["metrics"]["elapsed"]}s')
    print('下一步: 把 SVG 放到公司图纸目录下，作为伪标注用于 LoRA 微调')

else:
    print('跳过测试（RUN_TEST=False）')
    print('训练完成后改为 True')

AssertionError: 先训练模型: /workspace/production_3d/checkpoints_dinov2_lora/combined/best_dinov2_lora.pth

## Cell 11 · 批量处理公司图纸

In [ ]:
RUN_BATCH = False   # ← 单张测试通过后改为 True

if RUN_BATCH:
    # 扫描公司图纸目录
    company_images = [
        str(p) for p in Path(CFG.company_dir).rglob('*.png')
        if 'preprocessed' not in str(p)   # 跳过已有预处理结果
    ]
    company_images += [str(p) for p in Path(CFG.company_dir).rglob('*.jpg')]
    print(f'找到 {len(company_images)} 张公司图纸')

    stats = {'done': 0, 'error': 0}
    total_tokens = 0

    for img_path in tqdm(company_images, desc='批量伪标注', unit='张'):
        try:
            r = run_pseudo_label_pipeline(
                image_path    = img_path,
                cfg           = CFG,
                dinov2_model  = model,
                sam2_predictor= sam2_pred,
                vlm_client    = vlm_client,
                dry_run_vlm   = False,   # 批量时用真实 VLM
            )
            stats['done'] += 1
            # 统计 VLM token 消耗
            with open(os.path.join(CFG.pseudo_out_dir,
                      Path(img_path).stem, 'semantic_meta.json')) as f:
                meta = json.load(f)
            t = meta.get('_tokens', {})
            total_tokens += t.get('input', 0) + t.get('output', 0)

        except Exception as e:
            tqdm.write(f'ERROR {Path(img_path).name}: {e}')
            stats['error'] += 1

    print(f'\n批量完成  done={stats["done"]}  error={stats["error"]}')
    print(f'VLM 总 token 消耗: {total_tokens}  '
          f'(估算费用: ${total_tokens/1e6*3:.2f})')
    print(f'SVG 输出目录: {CFG.pseudo_out_dir}')

else:
    print('跳过批量处理（RUN_BATCH=False）')